In [2]:
import pandas as pd

import unicodedata
import re
def clean_text(s):
    # bỏ dấu tiếng Việt
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    
    # giữ lại chữ cái
    s = re.sub(r'[^a-zA-Z]', '', s)
    
    return s.lower()

s = "Tổng sản phẩm trong nước (GDP)!!!"
print(clean_text(s))





file_path = '../../../Bronze_Excel_Financial_Report/2023/IS_Thang/02-So-lieu-02-2023-1.xlsx'

excel_file = pd.ExcelFile(file_path)

all_sheets = excel_file.sheet_names

import_index = -1
export_index = -1

for i in range(len(all_sheets)):
    name = clean_text(all_sheets[i])
    if any(sheet in name for sheet in ['nk', 'nhapkhau']) and 'quy' not in name and 'gia' not in name:
        import_index = i
    if any(sheet in name for sheet in ['xk', 'xuatkhau']) and 'quy' not in name and 'gia' not in name:
        export_index = i

import_sheet = pd.read_excel(file_path, sheet_name= all_sheets[import_index], header= None)
export_sheet = pd.read_excel(file_path, sheet_name= all_sheets[export_index], header= None)

tongsanphamtrongnuocgdp


In [ ]:
def extract_intenational_ecommerce_data_sheet(sheet : pd.DataFrame, type : str):
    # xóa các row không cần thiết
    num_of_remove_row = 0
    for i in range(len(sheet)):
        num_of_remove_row += 1
        if isinstance(sheet.iloc[i, 0], str) and 'mathangchuyeu' in clean_text(sheet.iloc[i, 0]): break

    if type == 'import':
        sheet = sheet.iloc[num_of_remove_row:len(sheet) - 1, ::].reset_index(drop =True)
        
    else: sheet = sheet.iloc[num_of_remove_row::, ::].reset_index(drop =True)

    name_colums = ['product_name', 'luong_thang_nay', 'gia tri thang nay']
    #xoa cac cot kh can thiet
    sheet = sheet.iloc[::, [1, 2, 3]]
    sheet.columns = name_colums

    if type == 'import': 
        for i in range(len(sheet)):
            if 'oto' == clean_text(sheet.loc[i, 'product_name']):
                sheet.loc[i, 'product_name'] = 'Ô tô và linh kiện'
            if 'Trong đó: Nguyên chiếc(*)' in sheet.loc[i, 'product_name'] :
                sheet.loc[ i, 'product_name'] = 'Ô tô nguyên chiếc' 
        
    return sheet

In [ ]:
import_sheet
df = extract_intenational_ecommerce_data_sheet(export_sheet, 'export')

In [5]:
df

,product_name,luong_thang_nay,gia tri thang nay
0,Thủy sản,NaN,550
1,Rau quả,NaN,350
2,Hạt điều,30,171.155962
3,Cà phê,180,392.70184
4,Chè,8,13.194224
5,Hạt tiêu,28,85.641758
6,Gạo,430,230.589474
7,Sắn và sản phẩm của sắn,500,189.836918
8,Clanhke và xi măng,3200,136.533789
9,Dầu thô,280,203.456565


In [ ]:
df =extract_intenational_ecommerce_data_sheet(import_sheet, 'import')
df

,product_name,luong_thang_nay,gia tri thang nay
0,Thủy sản,NaN,250
1,Sữa và sản phẩm sữa,NaN,95
2,Rau quả,NaN,140
3,Hạt điều,150,211.315068
4,Ngô,750,251.823589
5,Thức ăn gia súc và NPL,NaN,450
6,Quặng và khoáng sản khác,1500,172.919558
7,Than đá,3700,613.490392
8,Dầu thô,700,424.513229
9,Xăng dầu,900,786.487875
